In [1]:
from lxml import etree
import pandas as pd

In [2]:
# File path
filename = "north-eastern-zone-latest.osm"

In [3]:

# Toll booth results
toll_booths = []

# Set up streaming XML parser
context = etree.iterparse(filename, events=('end',), tag='node')

In [4]:



for event, elem in context:
    tags = {tag.attrib['k']: tag.attrib['v'] for tag in elem.findall('tag')}
    
    # Skip nodes with no <tag>
    if not tags:
        elem.clear()
        continue
    
    # Check if it's a toll booth
    if ('barrier' in tags and tags['barrier'] == 'toll_booth') or \
       ('highway' in tags and tags['highway'] == 'toll_gantry'):
        
        node_data = {
            'id': elem.attrib.get('id'),
            'lat': elem.attrib.get('lat'),
            'lon': elem.attrib.get('lon'),
            'name': tags.get('name', None),
            **tags  # Unpack all tags
        }
        toll_booths.append(node_data)

    # Clear memory used by this node
    elem.clear()
    while elem.getprevious() is not None:
        del elem.getparent()[0]

# Convert to DataFrame
df = pd.DataFrame(toll_booths)

In [5]:
df.size

280

In [5]:
df.to_csv('../data/north_eastern_zone_booths.csv')